## Purpose and fixed analysis design

**RQ1.** To what extent are LLM judgements in a social-deduction setting
associated with persuasive properties of the discussion?

This is a **correlational, behavioural** analysis. The surrogate approximates
observable LLM choices; it does **not** establish causal influence or recover
the model's internal mechanism. Coefficients are associations, not effects.

- Unit: one row per (game, alternative), alternative = roster player or
  `No Werewolf`. Three stochastic runs per (model, game) are pooled into a
  vote-count target (sums to 3), not treated as independent rows.
- 13 persuasion features in 3 blocks (Strategy / Enriched / Combined) x 4
  representations (overall counts, overall rates, early/late counts,
  early/late rates).
- Outer CV: repeated 5-fold, grouped by the 36 player-group compositions,
  5 repeats (seeds 0-4). Inner CV: 3-fold, same grouping, for ridge/lasso
  penalty selection. Train-only standardisation; the `No Werewolf`
  alternative-specific constant is never penalised and its persuasion
  features stay exactly zero.
- Primary estimator: L2 conditional logit. Secondary: L1, for feature
  selection evidence only. No GBM, no binary classifiers, no interactions.
- Primary fidelity metric: held-out log loss against a null (ASC-only)
  surrogate. Secondary: stochastic top-choice agreement.
- Uncertainty: 2,000-resample (fidelity) / 1,000-resample (coefficients)
  clustered bootstrap over group compositions. This controls for
  composition-level dependence but NOT participant-level independence --
  22 of 60 players recur across compositions (Phase-0 finding).

In [1]:
import sys, time, json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

def find_repo_root(start=None, repo_name="masters_thesis_sdg"):
    current = (start or Path.cwd()).resolve()
    while True:
        if current.name == repo_name:
            return current
        if current.parent == current:
            raise FileNotFoundError(repo_name)
        current = current.parent

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "src"))

from utils_choice.model import run_validation_checks
from utils_choice.io import load_vote_tables, STOCHASTIC_RUNS
from utils_choice.rq1_features import (build_feature_table, ALL_FEATURES_13,
                                       STRATEGY_FEATURES, ENRICHED_FEATURES)
from utils_choice.rq1_cv import build_outer_folds, outer_splits, N_OUTER_FOLDS, OUTER_SEEDS
from utils_choice.rq1_model import (add_rate_columns, feature_columns, build_model_frame,
                                    run_nested_cv, per_game_avg_over_repeats, REPRESENTATIONS,
                                    BLOCKS, ASC_COL, choice_loss)
from utils_choice.rq1_pipeline import (run_null, run_fidelity_grid, bootstrap_paired_diff,
                                       bootstrap_group_mean, run_lofo, run_lasso_selection,
                                       combined_coefficients, bootstrap_coefficients)

MODEL_STAGE, PROMPT_DIR = "base", "prompt_v4"
ANALYSIS_ROOT = REPO_ROOT / "analysis"
TABLES_REL = Path(MODEL_STAGE) / "voting" / PROMPT_DIR / "vote_stability" / "tables"
ANNOT_ROOT = REPO_ROOT / "data" / "raw" / "lai2023"
ACC_ROOT = (REPO_ROOT / "data" / "processed" / "lai2023"
           / "accusation_transcripts" / "acc_targets")
IC_ROOT = (REPO_ROOT / "data" / "processed" / "lai2023"
          / "identity_claim_transcripts" / "ic_targets")

OUT_ROOT = ANALYSIS_ROOT / "cross_model" / MODEL_STAGE / "voting" / PROMPT_DIR / "predictive_v2"
TAB, FIG, DIA = OUT_ROOT / "tables", OUT_ROOT / "figures", OUT_ROOT / "diagnostics"
for d in (TAB, FIG, DIA):
    d.mkdir(parents=True, exist_ok=True)

MODELS = ["2B", "4B", "31B"]
BLOCK_NAMES = ("strategy", "enriched", "combined")

for msg in run_validation_checks():
    print("estimator validated:", msg)

estimator validated: recovers a known beta from simulated choices (max |error| = 0.018)
estimator validated: matches scikit-learn on the J=2 reduction (max |difference| = 1.6e-04)
estimator validated: analytic gradient matches the numerical one (error = 5.2e-05)
estimator validated: choice probabilities sum to 1 in every set (max error = 2.2e-16)
estimator validated: a strong L1 penalty zeroes coefficients (1 of 6 survive)


## Data validation

In [2]:
votes, games, roster = load_vote_tables(ANALYSIS_ROOT, TABLES_REL)
assert len(roster) == 191, f"expected 191 games, found {len(roster)}"
assert set(votes["model"].unique()) == {"2B", "4B", "31B"}, votes["model"].unique()
run_counts = votes.groupby(["model", "run_label"]).size().unstack()
print(run_counts)
assert (run_counts[list(STOCHASTIC_RUNS) + ["greedy_t0"]] == 191).all().all(), \
    "every model/run combination must cover all 191 games"
bad_status = votes[~votes["status"].isin(["player_vote", "circle_vote"])]
assert bad_status.empty, f"non-conforming vote rows: {len(bad_status)}"
print("data validation OK: 191 games, 3 models, 4 runs each, all votes conform")

# corpus annotation totals, re-verified against the Phase-0 audit (independent of
# any feature-construction choices made below -- this checks the raw sources)
from utils_choice.features import iter_annotation_games, ckey, PT_LABELS
_seen, _n_utt, _labels = set(), 0, Counter()
for source, session, game_id, dialogue in iter_annotation_games(ANNOT_ROOT):
    k = ckey(source, session, game_id)
    if k in _seen:
        continue
    _seen.add(k)
    _n_utt += len(dialogue)
    for u in dialogue:
        for a in (u.get("annotation") or []):
            if a in PT_LABELS:
                _labels[a] += 1
assert len(_seen) == 191, f"expected 191 retained ONUW games, found {len(_seen)}"
assert _n_utt == 24070, f"expected 24070 annotated utterances, found {_n_utt}"
assert _labels["Accusation"] == 3499, f"expected 3499 Accusation instances, found {_labels['Accusation']}"
assert _labels["Identity Declaration"] == 1359, \
    f"expected 1359 Identity Declaration instances, found {_labels['Identity Declaration']}"
print("corpus annotation totals reproduce exactly: 191 games, 24070 utterances, "
     f"3499 Accusation, 1359 Identity Declaration")

run_label  greedy_t0  run_1  run_2  run_3
model                                    
2B               191    191    191    191
31B              191    191    191    191
4B               191    191    191    191
data validation OK: 191 games, 3 models, 4 runs each, all votes conform
corpus annotation totals reproduce exactly: 191 games, 24070 utterances, 3499 Accusation, 1359 Identity Declaration


## Feature construction

In [3]:
feat_df, roster, nmaps, diag = build_feature_table(ANALYSIS_ROOT, TABLES_REL, ANNOT_ROOT,
                                                    ACC_ROOT, IC_ROOT)
feat_df, zero_report = add_rate_columns(feat_df)
print(f"player-game rows: {len(feat_df)} over {feat_df.key.nunique()} games")
print("target-level totals (pre speaker-match):", dict(diag["target_level_totals"]))
print("identity-claim item accounting:", dict(diag["ic_item_accounting"]))

unresolved = diag["unresolved_names"]
unresolved.to_csv(TAB / "unresolved_name_diagnostics.csv", index=False)
print(f"\nunresolved names logged: {len(unresolved)} rows -> unresolved_name_diagnostics.csv")
print(unresolved.groupby(["source", "raw_name"]).size().sort_values(ascending=False))

# aliases that the shared normalizer should recover must NOT appear as unresolved;
# the genuinely ambiguous Jordan1/Jordan2 case must still appear as unresolved
_resolved_aliases = {"Danieal", "Mitch", "Mithcell", "Chirs", "Jus"}
_bad = set(unresolved["raw_name"]) & _resolved_aliases
assert not _bad, f"these known-resolvable aliases leaked into the unresolved log: {_bad}"
assert "Jordan" in set(unresolved["raw_name"]), \
    "the ambiguous Jordan1/Jordan2 case should remain unresolved and documented"
print("\nalias-resolution assertions OK: Danieal/Mitch/Mithcell/Chirs/Jus resolved, "
     "Jordan correctly left unresolved (ambiguous: roster has both Jordan1 and Jordan2)")
print(f"\nzero-denominator rate rows: {len(zero_report)} "
      "(numerator asserted zero in every case -- see rq1_model.add_rate_columns)")
zero_report.to_csv(DIA / "zero_denominator_report.csv", index=False)

# conservation check: matched vs corpus target-level totals
made_ww = feat_df["werewolf_accusations_made"].sum()
made_dec = feat_df["deception_accusations_made"].sum()
recv_ww = feat_df["werewolf_accusations_received"].sum()
recv_dec = feat_df["deception_accusations_received"].sum()
tgt = diag["target_level_totals"]
print(f"\naccusation conservation: werewolf corpus={tgt['werewolf']} "
      f"made={made_ww:.0f} recv={recv_ww:.0f} (unresolved={tgt['werewolf']-made_ww:.0f} made / "
      f"{tgt['werewolf']-recv_ww:.0f} recv, all attributable to the ambiguous Jordan1/Jordan2 case)")
print(f"accusation conservation: deception corpus={tgt['deception']} "
      f"made={made_dec:.0f} recv={recv_dec:.0f} (unresolved={tgt['deception']-made_dec:.0f} made / "
      f"{tgt['deception']-recv_dec:.0f} recv)")

# assertions: no ground-truth fields, no turns as feature, non-negative
for f in ALL_FEATURES_13:
    assert (feat_df[f] >= 0).all()
assert "n_turns" not in ALL_FEATURES_13 and "turns" not in ALL_FEATURES_13
print("\nfeature assertions OK: all non-negative, no ground-truth fields, "
      "turns excluded from the feature list")

player-game rows: 864 over 191 games
target-level totals (pre speaker-match): {'werewolf': 1200, 'deception': 640}
identity-claim item accounting: {'items': 1359, 'resolved': 1317, 'multi': 44, 'empty': 41, 'unknown': 1}

unresolved names logged: 158 rows -> unresolved_name_diagnostics.csv
source                  raw_name
original_speaker        Jordan      95
accusation_target       Jordan      22
identity_claim_speaker  Jordan      15
accusation_accuser      Jordan      13
original_speaker        Matt         9
                        Kayla        3
identity_claim_speaker  Kayla        1
dtype: int64

alias-resolution assertions OK: Danieal/Mitch/Mithcell/Chirs/Jus resolved, Jordan correctly left unresolved (ambiguous: roster has both Jordan1 and Jordan2)

zero-denominator rate rows: 275 (numerator asserted zero in every case -- see rq1_model.add_rate_columns)

accusation conservation: werewolf corpus=1200 made=1198 recv=1189 (unresolved=2 made / 11 recv, all attributable to the ambi

## Feature support and diagnostics

In [4]:
support_rows = []
for feat in ALL_FEATURES_13:
    x = feat_df[feat]
    support_rows.append({"feature": feat, "block": ("strategy" if feat in STRATEGY_FEATURES
                                                     else "enriched"),
                         "total": float(x.sum()), "player_games_gt0": int((x > 0).sum()),
                         "pct_player_games_gt0": round(100 * (x > 0).mean(), 2),
                         "distinct_games_gt0": int(feat_df.loc[x > 0, "key"].nunique()),
                         "mean": round(x.mean(), 3), "median": float(x.median()),
                         "sd": round(x.std(), 3), "max": float(x.max())})
support_tab = pd.DataFrame(support_rows)
support_tab.to_csv(TAB / "feature_support_final.csv", index=False)
print(support_tab.to_string(index=False))

                       feature    block  total  player_games_gt0  pct_player_games_gt0  distinct_games_gt0  mean  median    sd  max
                    accusation strategy 3474.0               748                 86.57                 190 4.021     3.0 3.389 22.0
                       defense strategy 3260.0               715                 82.75                 187 3.773     3.0 3.808 24.0
                 interrogation strategy 4084.0               802                 92.82                 190 4.727     4.0 3.713 24.0
                      evidence strategy 2221.0               730                 84.49                 189 2.571     2.0 2.169 13.0
          identity_declaration strategy 1343.0               637                 73.73                 189 1.554     1.0 1.504  9.0
               call_for_action strategy 1390.0               554                 64.12                 180 1.609     1.0 1.888 11.0
     werewolf_accusations_made enriched 1198.0               452            

## Grouped nested-CV setup

In [5]:
fold_df = build_outer_folds(roster, n_folds=N_OUTER_FOLDS, seeds=OUTER_SEEDS)
fold_df.to_csv(TAB / "group_cv_assignments.csv", index=False)
print(f"fold assignments: {len(fold_df)} rows "
      f"({fold_df.repeat.nunique()} repeats x {fold_df.key.nunique()} games)")
print(f"unique group compositions: {fold_df.composition_id.nunique()}")
print(fold_df.groupby(["repeat", "fold"]).size().unstack())
_, comp_of_key_r0 = outer_splits(fold_df, repeat=0)
assert set(comp_of_key_r0) == set(roster), "fold composition map must cover every game"
print("CV assertions OK: 191 games x 5 repeats, 36 compositions, no cross-fold composition split "
      "(enforced inside build_outer_folds)")

fold assignments: 955 rows (5 repeats x 191 games)
unique group compositions: 36
fold     0   1   2   3   4
repeat                    
0       39  38  38  37  39
1       37  39  38  38  39
2       41  47  35  32  36
3       40  37  41  36  37
4       38  38  40  33  42
CV assertions OK: 191 games x 5 repeats, 36 compositions, no cross-fold composition split (enforced inside build_outer_folds)


## Build the per-model long frames (all representations share one feature table)

In [6]:
frames = {m: build_model_frame(feat_df, roster, comp_of_key_r0, votes, m) for m in MODELS}
for m, fr in frames.items():
    assert fr["key"].nunique() == 191
    tot = fr.groupby("key")["count"].sum()
    assert np.allclose(tot.values, 3.0), f"{m}: vote counts must sum to 3 per game"
    circle_rows = fr[fr[ASC_COL] == 1.0]
    assert (circle_rows[ALL_FEATURES_13] == 0).all().all(), \
        f"{m}: the No Werewolf alternative must carry zero persuasion information"
    print(f"{m}: {len(fr)} ballot rows over {fr.key.nunique()} games "
         f"(mean {fr.groupby('key').size().mean():.2f} alternatives/game)")
print("assertion OK: No Werewolf alternative is zero on every persuasion feature, all models")

# spot check: fitted choice probabilities sum to 1 within every ballot (also
# covered at the estimator level by run_validation_checks() above)
_spot = frames["2B"]
_spot_train, _spot_test = _spot.iloc[:len(_spot) // 2], _spot.iloc[len(_spot) // 2:]
_spot_cols = feature_columns("count_overall", "combined")
from utils_choice.rq1_model import fit_predict_penalized as _fpp
_q, _ = _fpp(_spot_train, _spot_test, _spot_cols, "l2", inner_seed=0)[:2]
_sums = _spot_test.assign(q=_q).groupby("key")["q"].sum()
assert np.allclose(_sums.values, 1.0), "predicted probabilities must sum to 1 within each ballot"
assert np.isfinite(_q).all(), "predicted probabilities must be finite"
print("assertion OK: predicted choice probabilities sum to 1 and are finite (spot check)")

2B: 1055 ballot rows over 191 games (mean 5.52 alternatives/game)
4B: 1055 ballot rows over 191 games (mean 5.52 alternatives/game)
31B: 1055 ballot rows over 191 games (mean 5.52 alternatives/game)
assertion OK: No Werewolf alternative is zero on every persuasion feature, all models
assertion OK: predicted choice probabilities sum to 1 and are finite (spot check)


## Null model / reference points

In [7]:
t0 = time.time()
null_results = {m: run_null(frames[m], fold_df) for m in MODELS}
print(f"null models fit in {time.time() - t0:.1f}s")

ref_rows = []
for m, fr in frames.items():
    pg_null_avg = per_game_avg_over_repeats(null_results[m][0])
    shares = fr.assign(share=fr["count"] / 3.0)
    best_possible = shares.groupby("key")["share"].max().mean()
    sizes = fr.groupby("key").size()
    uniform_ll = float(np.log(sizes).mean())
    ref_rows.append({"model": m, "null_mean_loss": float(pg_null_avg["loss"].mean()),
                     "null_mean_agreement": float(pg_null_avg["agreement"].mean()),
                     "best_possible_deterministic_agreement": float(best_possible),
                     "uniform_choice_loss_descriptive": uniform_ll,
                     "n_games": fr.key.nunique()})
reference_points = pd.DataFrame(ref_rows)
reference_points.to_csv(TAB / "reference_points.csv", index=False)
print(reference_points.to_string(index=False))

null models fit in 1.1s
model  null_mean_loss  null_mean_agreement  best_possible_deterministic_agreement  uniform_choice_loss_descriptive  n_games
   2B        1.701788             0.228621                               0.788831                         1.699497      191
   4B        1.641750             0.212856                               0.815009                         1.699497      191
  31B        1.692473             0.198429                               0.849913                         1.699497      191


## Ridge surrogate fidelity grid (3 models x 4 representations x 3 blocks)

In [8]:
t0 = time.time()
pg_grid, fold_grid = run_fidelity_grid(frames, fold_df, blocks=BLOCK_NAMES,
                                       representations=REPRESENTATIONS)
print(f"fidelity grid fit in {(time.time() - t0) / 60:.1f} min "
     f"({pg_grid.groupby(['model','representation','block']).ngroups} cells)")
fold_grid.to_csv(TAB / "ridge_fidelity_fold_level.csv", index=False)

summary_rows = []
for (m, r, b), g in pg_grid.groupby(["model", "representation", "block"]):
    avg = per_game_avg_over_repeats(g)
    null_avg = per_game_avg_over_repeats(null_results[m][0])
    diff = bootstrap_paired_diff(avg, null_avg, metric="loss", label_a="surrogate", label_b="null")
    summary_rows.append({"model": m, "representation": r, "block": b,
                        "mean_loss": float(avg["loss"].mean()),
                        "mean_agreement": float(avg["agreement"].mean()),
                        "null_loss": float(null_avg["loss"].mean()),
                        "delta_loss_vs_null": diff["mean_diff"],
                        "delta_loss_ci_lo": diff["ci_lo"], "delta_loss_ci_hi": diff["ci_hi"],
                        "improves_over_null": diff["excludes_zero"] and diff["mean_diff"] > 0})
ridge_fidelity_summary = pd.DataFrame(summary_rows)
ridge_fidelity_summary.to_csv(TAB / "ridge_fidelity_summary.csv", index=False)
print(ridge_fidelity_summary.round(4).to_string(index=False))

fidelity grid fit in 5.8 min (36 cells)


model representation    block  mean_loss  mean_agreement  null_loss  delta_loss_vs_null  delta_loss_ci_lo  delta_loss_ci_hi  improves_over_null
   2B  count_overall combined     1.4308          0.4471     1.7018              0.2710            0.1821            0.3575                True
   2B  count_overall enriched     1.4363          0.4551     1.7018              0.2655            0.1840            0.3467                True
   2B  count_overall strategy     1.6892          0.2133     1.7018              0.0126           -0.0080            0.0361               False
   2B count_temporal combined     1.4441          0.4597     1.7018              0.2577            0.1800            0.3376                True
   2B count_temporal enriched     1.4506          0.4468     1.7018              0.2512            0.1697            0.3310                True
   2B count_temporal strategy     1.6664          0.2520     1.7018              0.0354            0.0043            0.0728             

## Count vs rate and temporal sensitivity -- paired comparisons

In [9]:
def avg_of(model, repr_, block):
    g = pg_grid[(pg_grid.model == model) & (pg_grid.representation == repr_)
               & (pg_grid.block == block)]
    return per_game_avg_over_repeats(g)

paired_specs = [
    ("Combined vs Strategy, overall counts", "count_overall", "combined", "count_overall", "strategy"),
    ("Combined vs Strategy, overall rates", "rate_overall", "combined", "rate_overall", "strategy"),
    ("Enriched vs Strategy, overall counts", "count_overall", "enriched", "count_overall", "strategy"),
    ("Counts vs Rates, Combined", "rate_overall", "combined", "count_overall", "combined"),
    ("Temporal counts vs overall counts, Combined", "count_temporal", "combined", "count_overall", "combined"),
    ("Temporal rates vs overall rates, Combined", "rate_temporal", "combined", "rate_overall", "combined"),
]

paired_rows = []
for m in MODELS:
    for label, r_b, b_b, r_a, b_a in paired_specs:
        pg_b, pg_a = avg_of(m, r_b, b_b), avg_of(m, r_a, b_a)
        d_loss = bootstrap_paired_diff(pg_a, pg_b, metric="loss", label_a=b_a, label_b=b_b)
        d_agree = bootstrap_paired_diff(pg_a, pg_b, metric="agreement", label_a=b_a, label_b=b_b)
        paired_rows.append({"model": m, "comparison": label,
                           "loss_mean_diff": d_loss["mean_diff"], "loss_ci_lo": d_loss["ci_lo"],
                           "loss_ci_hi": d_loss["ci_hi"], "loss_excludes_zero": d_loss["excludes_zero"],
                           "agreement_mean_diff": d_agree["mean_diff"],
                           "agreement_ci_lo": d_agree["ci_lo"], "agreement_ci_hi": d_agree["ci_hi"]})
ridge_fidelity_paired_comparisons = pd.DataFrame(paired_rows)
ridge_fidelity_paired_comparisons.to_csv(TAB / "ridge_fidelity_paired_comparisons.csv", index=False)
print(ridge_fidelity_paired_comparisons.round(4).to_string(index=False))

temporal_rows = []
for m in MODELS:
    for block in BLOCK_NAMES:
        for kind, repr_temp, repr_overall in [("counts", "count_temporal", "count_overall"),
                                               ("rates", "rate_temporal", "rate_overall")]:
            pg_t, pg_o = avg_of(m, repr_temp, block), avg_of(m, repr_overall, block)
            d = bootstrap_paired_diff(pg_o, pg_t, metric="loss", label_a="overall", label_b="temporal")
            temporal_rows.append({"model": m, "block": block, "kind": kind,
                                 "temporal_loss": float(pg_t["loss"].mean()),
                                 "overall_loss": float(pg_o["loss"].mean()),
                                 "delta_temporal_minus_overall": d["mean_diff"],
                                 "ci_lo": d["ci_lo"], "ci_hi": d["ci_hi"],
                                 "excludes_zero": d["excludes_zero"]})
temporal_fidelity_summary = pd.DataFrame(temporal_rows)
temporal_fidelity_summary.to_csv(TAB / "temporal_fidelity_summary.csv", index=False)
print(temporal_fidelity_summary.round(4).to_string(index=False))

model                                  comparison  loss_mean_diff  loss_ci_lo  loss_ci_hi  loss_excludes_zero  agreement_mean_diff  agreement_ci_lo  agreement_ci_hi
   2B        Combined vs Strategy, overall counts         -0.2584     -0.3405     -0.1690                True               0.2339           0.1830           0.2902
   2B         Combined vs Strategy, overall rates         -0.2276     -0.3076     -0.1260                True               0.2168           0.1659           0.2781
   2B        Enriched vs Strategy, overall counts         -0.2529     -0.3292     -0.1716                True               0.2419           0.1893           0.3006
   2B                   Counts vs Rates, Combined          0.0505     -0.0016      0.0991               False              -0.0140          -0.0418           0.0209
   2B Temporal counts vs overall counts, Combined          0.0133     -0.0290      0.0505               False               0.0126          -0.0108           0.0386
   2B   Te

model    block   kind  temporal_loss  overall_loss  delta_temporal_minus_overall   ci_lo   ci_hi  excludes_zero
   2B strategy counts         1.6664        1.6892                       -0.0228 -0.0456 -0.0020           True
   2B strategy  rates         1.7130        1.7088                        0.0041 -0.0128  0.0235          False
   2B enriched counts         1.4506        1.4363                        0.0143 -0.0228  0.0502          False
   2B enriched  rates         1.5057        1.4780                        0.0277 -0.0178  0.0685          False
   2B combined counts         1.4441        1.4308                        0.0133 -0.0290  0.0505          False
   2B combined  rates         1.5032        1.4812                        0.0220 -0.0267  0.0643          False
   4B strategy counts         1.6118        1.6299                       -0.0181 -0.0410  0.0038          False
   4B strategy  rates         1.6396        1.6501                       -0.0104 -0.0227  0.0005        

## Sparse L1 sensitivity (Combined block, overall counts / rates only)

In [10]:
t0 = time.time()
lasso_counts_rows, lasso_rates_rows = [], []
for m in MODELS:
    tab_c, _ = run_lasso_selection(frames[m], fold_df, "count_overall", block="combined")
    tab_c.insert(0, "model", m); lasso_counts_rows.append(tab_c)
    tab_r, _ = run_lasso_selection(frames[m], fold_df, "rate_overall", block="combined")
    tab_r.insert(0, "model", m); lasso_rates_rows.append(tab_r)
lasso_selection_counts = pd.concat(lasso_counts_rows, ignore_index=True)
lasso_selection_rates = pd.concat(lasso_rates_rows, ignore_index=True)
lasso_selection_counts.to_csv(TAB / "lasso_selection_counts.csv", index=False)
lasso_selection_rates.to_csv(TAB / "lasso_selection_rates.csv", index=False)
print(f"L1 sensitivity fit in {(time.time() - t0) / 60:.1f} min")
print(lasso_selection_counts.round(3).to_string(index=False))

L1 sensitivity fit in 1.3 min
model                        feature representation    block  n_outer_models  selection_freq  positive_selection_freq  negative_selection_freq  median_nonzero_coef  median_selected_lambda
   2B                     accusation  count_overall combined              25            0.20                     0.00                     0.20               -0.037                    10.0
   2B                        defense  count_overall combined              25            1.00                     1.00                     0.00                0.111                    10.0
   2B                  interrogation  count_overall combined              25            0.92                     0.00                     0.92               -0.061                    10.0
   2B                       evidence  count_overall combined              25            0.60                     0.00                     0.60               -0.030                    10.0
   2B           identity_decla

## Combined-model ridge coefficients (overall counts, overall rates)

In [11]:
t0 = time.time()
coef_counts_rows, coef_rates_rows = [], []
boot_counts_rows, boot_rates_rows = [], []
boot_failures = []
for m in MODELS:
    tab_c, pen_c = combined_coefficients(frames[m], "count_overall", block="combined")
    tab_c.insert(0, "model", m); coef_counts_rows.append(tab_c)
    bc, fails_c = bootstrap_coefficients(frames[m], "count_overall", "combined", pen_c)
    bc.insert(0, "model", m); boot_counts_rows.append(bc)
    if len(fails_c):
        fails_c["model"] = m; fails_c["representation"] = "count_overall"; boot_failures.append(fails_c)

    tab_r, pen_r = combined_coefficients(frames[m], "rate_overall", block="combined")
    tab_r.insert(0, "model", m); coef_rates_rows.append(tab_r)
    br, fails_r = bootstrap_coefficients(frames[m], "rate_overall", "combined", pen_r)
    br.insert(0, "model", m); boot_rates_rows.append(br)
    if len(fails_r):
        fails_r["model"] = m; fails_r["representation"] = "rate_overall"; boot_failures.append(fails_r)

ridge_coefficients_counts = pd.concat(coef_counts_rows, ignore_index=True)
ridge_coefficients_rates = pd.concat(coef_rates_rows, ignore_index=True)
ridge_coefficients_counts.to_csv(TAB / "ridge_coefficients_counts.csv", index=False)
ridge_coefficients_rates.to_csv(TAB / "ridge_coefficients_rates.csv", index=False)

ridge_coefficient_bootstrap_counts = (pd.concat(boot_counts_rows, ignore_index=True)
                                      .merge(ridge_coefficients_counts[["model","feature","beta","odds_ratio"]],
                                             on=["model","feature"]))
ridge_coefficient_bootstrap_rates = (pd.concat(boot_rates_rows, ignore_index=True)
                                     .merge(ridge_coefficients_rates[["model","feature","beta","odds_ratio"]],
                                            on=["model","feature"]))
ridge_coefficient_bootstrap_counts.to_csv(TAB / "ridge_coefficient_bootstrap_counts.csv", index=False)
ridge_coefficient_bootstrap_rates.to_csv(TAB / "ridge_coefficient_bootstrap_rates.csv", index=False)
if boot_failures:
    pd.concat(boot_failures, ignore_index=True).to_csv(DIA / "coefficient_bootstrap_failures.csv", index=False)
print(f"coefficient fit + bootstrap done in {(time.time() - t0) / 60:.1f} min, "
     f"{sum(len(f) for f in boot_failures)} bootstrap failures")
print("\n-- counts --")
print(ridge_coefficient_bootstrap_counts.round(3).to_string(index=False))
print("\n-- rates --")
print(ridge_coefficient_bootstrap_rates.round(3).to_string(index=False))

coefficient fit + bootstrap done in 9.2 min, 0 bootstrap failures

-- counts --
model                        feature                         column  beta_ci_lo  beta_ci_hi  odds_ratio_ci_lo  odds_ratio_ci_hi  n_boot_used  n_failures   beta  odds_ratio
   2B                     accusation                     accusation      -0.235       0.178             0.790             1.194         1000           0 -0.056       0.946
   2B                        defense                        defense       0.089       0.402             1.093             1.495         1000           0  0.239       1.270
   2B                  interrogation                  interrogation      -0.291       0.050             0.748             1.051         1000           0 -0.096       0.908
   2B                       evidence                       evidence      -0.246       0.100             0.782             1.105         1000           0 -0.061       0.941
   2B           identity_declaration           identity_decl

## LOFO importance (Combined block, overall counts / rates only)

In [12]:
t0 = time.time()
lofo_counts_rows, lofo_rates_rows = [], []
for m in MODELS:
    tab_c, _ = run_lofo(frames[m], fold_df, "count_overall", block="combined")
    tab_c.insert(0, "model", m); lofo_counts_rows.append(tab_c)
    tab_r, _ = run_lofo(frames[m], fold_df, "rate_overall", block="combined")
    tab_r.insert(0, "model", m); lofo_rates_rows.append(tab_r)
lofo_counts = pd.concat(lofo_counts_rows, ignore_index=True)
lofo_rates = pd.concat(lofo_rates_rows, ignore_index=True)
lofo_counts.to_csv(TAB / "lofo_counts.csv", index=False)
lofo_rates.to_csv(TAB / "lofo_rates.csv", index=False)
print(f"LOFO fit in {(time.time() - t0) / 60:.1f} min")
print(lofo_counts.round(4).to_string(index=False))

LOFO fit in 14.4 min
model                        feature representation    block  lofo_importance   ci_lo   ci_hi  excludes_zero
   2B                     accusation  count_overall combined          -0.0007 -0.0032  0.0014          False
   2B                        defense  count_overall combined           0.0106  0.0025  0.0204           True
   2B                  interrogation  count_overall combined          -0.0011 -0.0076  0.0055          False
   2B                       evidence  count_overall combined          -0.0023 -0.0065  0.0025          False
   2B           identity_declaration  count_overall combined          -0.0009 -0.0165  0.0104          False
   2B                call_for_action  count_overall combined           0.0012 -0.0045  0.0068          False
   2B      werewolf_accusations_made  count_overall combined           0.0214  0.0073  0.0346           True
   2B     deception_accusations_made  count_overall combined          -0.0045 -0.0084 -0.0013           Tru

## Figures

The four thesis figures are built from the CSVs already saved under
`predictive_v2/tables/`, not from anything held in memory. To iterate on
figure code without refitting: **restart the kernel, run the setup cell, run
the figure-inputs cell below, then the figure cell.** That path takes seconds.

In [13]:
# --- figure inputs -----------------------------------------------------
# The figures below depend ONLY on tables already written to disk by the
# sections above, so they can be regenerated without refitting anything.
#
# Fast path for iterating on figure code (seconds, not ~20 min):
#     restart kernel -> run the setup cell -> run THIS cell -> run the
#     figure cell.
#
# Running the notebook straight through also passes through here harmlessly:
# these are the same objects, round-tripped via their saved CSVs.

FIGURE_INPUTS = ["ridge_fidelity_summary",
                 "ridge_coefficient_bootstrap_counts",
                 "ridge_coefficient_bootstrap_rates",
                 "lofo_counts",
                 "lofo_rates"]

_missing = [n for n in FIGURE_INPUTS if not (TAB / f"{n}.csv").exists()]
if _missing:
    raise FileNotFoundError(
        "No saved table for: " + ", ".join(_missing)
        + f"\nRun the analysis sections once to populate {TAB}.")

ridge_fidelity_summary = pd.read_csv(TAB / "ridge_fidelity_summary.csv")
ridge_coefficient_bootstrap_counts = pd.read_csv(
    TAB / "ridge_coefficient_bootstrap_counts.csv")
ridge_coefficient_bootstrap_rates = pd.read_csv(
    TAB / "ridge_coefficient_bootstrap_rates.csv")
lofo_counts = pd.read_csv(TAB / "lofo_counts.csv")
lofo_rates = pd.read_csv(TAB / "lofo_rates.csv")

print(f"figure inputs loaded from {TAB.relative_to(REPO_ROOT)}")
for _n in FIGURE_INPUTS:
    print(f"  {_n:36s} {len(globals()[_n]):3d} rows")

figure inputs loaded from analysis\cross_model\base\voting\prompt_v4\predictive_v2\tables
  ridge_fidelity_summary                36 rows
  ridge_coefficient_bootstrap_counts    39 rows
  ridge_coefficient_bootstrap_rates     39 rows
  lofo_counts                           39 rows
  lofo_rates                            39 rows


In [14]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Typography, sized up so the saved PNGs stay legible on screen and when
# the figure is scaled down to text width in the thesis.
plt.rcParams.update({
    "font.size": 13,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 13,
    "legend.fontsize": 12,
    "legend.title_fontsize": 13,
})

# ---------------------------------------------------------------------
# Shared plotting configuration
# ---------------------------------------------------------------------

MODEL_ORDER = ["2B", "4B", "31B"]
MODEL_LABELS = {
    "2B": "E2B",
    "4B": "E4B",
    "31B": "31B",
}

BLOCK_ORDER = ["strategy", "enriched", "combined"]
BLOCK_LABELS = {
    "strategy": "Strategy",
    "enriched": "Content and targets",
    "combined": "Combined",
}

# Okabe–Ito color-blind-friendly palette
MODEL_COLORS = {
    "2B": "#0072B2",   # blue
    "4B": "#D55E00",   # vermillion
    "31B": "#009E73",  # bluish green
}

BLOCK_COLORS = {
    "strategy": "#0072B2",   # blue
    "enriched": "#E69F00",   # orange
    "combined": "#009E73",   # bluish green
}

FEATURE_LABELS = {
    "accusation": "Accusation",
    "defense": "Defense",
    "interrogation": "Interrogation",
    "evidence": "Evidence",
    "identity_declaration": "Identity Declaration",
    "call_for_action": "Call for Action",
    "werewolf_accusations_made": "Werewolf accusations made",
    "deception_accusations_made": "Deception accusations made",
    "werewolf_accusations_received": "Werewolf accusations received",
    "deception_accusations_received": "Deception accusations received",
    "claims_werewolf": "Claims Werewolf",
    "claims_tanner": "Claims Tanner",
    "claims_night_action_role": "Claims night-action role",
}

FEATURES = ALL_FEATURES_13
FEATURE_LABEL_LIST = [FEATURE_LABELS.get(f, f.replace("_", " ")) for f in FEATURES]

# Strategy contains the first six features.
STRATEGY_END = 6

# Vertical distance between consecutive feature rows, in y units. The three
# model markers for a feature sit within +/- MODEL_OFFSET of its row centre,
# so raising GROUP_SPACING widens the gap BETWEEN features without crowding
# the three models within a feature.
GROUP_SPACING = 1.8
MODEL_OFFSET = 0.34


# ---------------------------------------------------------------------
# Figure A: fidelity relative to fitted null
# ---------------------------------------------------------------------

figA = ridge_fidelity_summary[
    ridge_fidelity_summary.representation.isin(
        ["count_overall", "rate_overall"]
    )
].copy()

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.4), sharey=True)

repr_titles = {
    "count_overall": "Counts",
    "rate_overall": "Rates",
}

for ax, repr_ in zip(axes, ["count_overall", "rate_overall"]):
    sub = figA[figA.representation == repr_]

    width = 0.24
    x = np.arange(len(MODEL_ORDER))

    for i, block in enumerate(BLOCK_ORDER):
        s = (
            sub[sub.block == block]
            .set_index("model")
            .reindex(MODEL_ORDER)
        )

        ax.bar(
            x + (i - 1) * width,
            s["delta_loss_vs_null"],
            width,
            label=BLOCK_LABELS[block],
            color=BLOCK_COLORS[block],
            yerr=[
                s["delta_loss_vs_null"] - s["delta_loss_ci_lo"],
                s["delta_loss_ci_hi"] - s["delta_loss_vs_null"],
            ],
            capsize=4,
        )

    ax.axhline(0, color="black", lw=0.8)

    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_LABELS[m] for m in MODEL_ORDER])

    ax.set_title(repr_titles[repr_])
    ax.set_xlabel("Gemma variant")

axes[0].set_ylabel(
    r"Improvement over null ($\Delta$ log loss)"
    "\n"
    r"(positive values favor the surrogate)"
)

axes[0].legend(
    title="Feature set",
    loc="upper left",
    fontsize=8,
)

fig.tight_layout()
fig.savefig(
    FIG / "figureA_fidelity_vs_null.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close(fig)


# ---------------------------------------------------------------------
# Shared limits for coefficient Figures B and C
# ---------------------------------------------------------------------

coef_ci_min = min(
    ridge_coefficient_bootstrap_counts["beta_ci_lo"].min(),
    ridge_coefficient_bootstrap_rates["beta_ci_lo"].min(),
)

coef_ci_max = max(
    ridge_coefficient_bootstrap_counts["beta_ci_hi"].max(),
    ridge_coefficient_bootstrap_rates["beta_ci_hi"].max(),
)

coef_range = coef_ci_max - coef_ci_min
coef_margin = 0.07 * coef_range

COEF_XLIM = (
    coef_ci_min - coef_margin,
    coef_ci_max + coef_margin,
)


# ---------------------------------------------------------------------
# Figures B/C: Combined ridge coefficients
# ---------------------------------------------------------------------

def coef_figure(tab, title, path):
    fig, ax = plt.subplots(figsize=(9.5, 12.5))

    y0 = np.arange(len(FEATURES)) * GROUP_SPACING

    # Small offsets keep the three models visually distinct.
    offsets = {
        "2B": -MODEL_OFFSET,
        "4B": 0.0,
        "31B": MODEL_OFFSET,
    }

    for m in MODEL_ORDER:
        s = (
            tab[tab.model == m]
            .set_index("feature")
            .reindex(FEATURES)
        )

        y = y0 + offsets[m]

        ax.errorbar(
            s["beta"],
            y,
            xerr=[
                s["beta"] - s["beta_ci_lo"],
                s["beta_ci_hi"] - s["beta"],
            ],
            fmt="o",
            color=MODEL_COLORS[m],
            label=MODEL_LABELS[m],
            capsize=4,
            markersize=7,
            linewidth=1.9,
        )

    # Zero coefficient
    ax.axvline(0, color="black", lw=0.9)

    # Separate Strategy from Content and targets
    ax.axhline(
        (STRATEGY_END - 0.5) * GROUP_SPACING,
        color="0.75",
        lw=0.9,
        linestyle="--",
    )

    ax.set_yticks(y0)
    ax.set_yticklabels(FEATURE_LABEL_LIST)
    ax.invert_yaxis()

    # Same x-axis for counts and rates to facilitate visual comparison
    ax.set_xlim(*COEF_XLIM)

    ax.set_xlabel(
        r"Standardized coefficient $\beta$ (95% bootstrap CI)"
    )
    ax.set_title(title)

    ax.legend(
        title="Gemma variant",
        loc="upper right",
        frameon=True,
    )

    fig.tight_layout()
    fig.savefig(
        path,
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)


coef_figure(
    ridge_coefficient_bootstrap_counts,
    "Count representation",
    FIG / "figureB_coefficients_counts.png",
)

coef_figure(
    ridge_coefficient_bootstrap_rates,
    "Rate representation",
    FIG / "figureC_coefficients_rates.png",
)


# ---------------------------------------------------------------------
# Figure D: leave-one-feature-out importance
# ---------------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14.5, 12.5),
    sharey=True,
)

for ax, tab, title in zip(
    axes,
    [lofo_counts, lofo_rates],
    ["Counts", "Rates"],
):
    y0 = np.arange(len(FEATURES)) * GROUP_SPACING

    offsets = {
        "2B": -MODEL_OFFSET,
        "4B": 0.0,
        "31B": MODEL_OFFSET,
    }

    for m in MODEL_ORDER:
        s = (
            tab[tab.model == m]
            .set_index("feature")
            .reindex(FEATURES)
        )

        y = y0 + offsets[m]

        ax.errorbar(
            s["lofo_importance"],
            y,
            xerr=[
                s["lofo_importance"] - s["ci_lo"],
                s["ci_hi"] - s["lofo_importance"],
            ],
            fmt="o",
            color=MODEL_COLORS[m],
            label=MODEL_LABELS[m],
            capsize=4,
            markersize=7,
            linewidth=1.9,
        )

    ax.axvline(0, color="black", lw=0.9)

    # Separate Strategy from Content and targets
    ax.axhline(
        (STRATEGY_END - 0.5) * GROUP_SPACING,
        color="0.75",
        lw=0.9,
        linestyle="--",
    )

    ax.set_yticks(y0)
    ax.set_yticklabels(FEATURE_LABEL_LIST)
    ax.set_xlabel(
        r"Increase in held-out log loss after feature removal"
    )
    ax.set_title(title)

# One inversion only: the panels share a y-axis, so inverting inside the
# loop above would flip it twice and silently undo itself.
axes[0].invert_yaxis()

axes[0].legend(
    title="Gemma variant",
    loc="best",
)

fig.tight_layout()
fig.savefig(
    FIG / "figureD_lofo.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close(fig)


# ---------------------------------------------------------------------
# Output check
# ---------------------------------------------------------------------

print("Figures saved:")
for p in sorted(FIG.glob("*.png")):
    print(" ", p.relative_to(REPO_ROOT))

Figures saved:
  analysis\cross_model\base\voting\prompt_v4\predictive_v2\figures\figureA_fidelity_vs_null.png
  analysis\cross_model\base\voting\prompt_v4\predictive_v2\figures\figureB_coefficients_counts.png
  analysis\cross_model\base\voting\prompt_v4\predictive_v2\figures\figureC_coefficients_rates.png
  analysis\cross_model\base\voting\prompt_v4\predictive_v2\figures\figureD_lofo.png


## Compact conclusions / diagnostics

In [15]:
print("=== Phase 1 run complete ===")
print(f"output directory: {OUT_ROOT.relative_to(REPO_ROOT)}")
print(f"\nunresolved names: {len(unresolved)} (Jordan1/Jordan2 ambiguity + a small number of "
     "off-roster bystanders -- see unresolved_name_diagnostics.csv)")
print("\nfidelity summary (combined block, overall representations):")
print(ridge_fidelity_summary[(ridge_fidelity_summary.block == "combined")
                             & (ridge_fidelity_summary.representation.isin(
                                 ["count_overall", "rate_overall"]))]
     [["model", "representation", "mean_loss", "null_loss", "delta_loss_vs_null",
       "improves_over_null"]].round(4).to_string(index=False))
print("\nsaved tables:", sorted(p.name for p in TAB.glob("*.csv")))
print("saved figures:", sorted(p.name for p in FIG.glob("*.png")))

=== Phase 1 run complete ===
output directory: analysis\cross_model\base\voting\prompt_v4\predictive_v2

unresolved names: 158 (Jordan1/Jordan2 ambiguity + a small number of off-roster bystanders -- see unresolved_name_diagnostics.csv)

fidelity summary (combined block, overall representations):
model representation  mean_loss  null_loss  delta_loss_vs_null  improves_over_null
   2B  count_overall     1.4308     1.7018              0.2710                True
   2B   rate_overall     1.4812     1.7018              0.2205                True
  31B  count_overall     1.4941     1.6925              0.1984                True
  31B   rate_overall     1.5669     1.6925              0.1256                True
   4B  count_overall     1.3341     1.6418              0.3076                True
   4B   rate_overall     1.3212     1.6418              0.3205                True

saved tables: ['feature_support_final.csv', 'group_cv_assignments.csv', 'lasso_selection_counts.csv', 'lasso_selection_ra